In [6]:
import polars as pl
from plotly import express as px, graph_objects as go

In [7]:
textos = pl.read_parquet('../database/textos/*.parquet').select('id', 'texto')
metadata = pl.read_parquet('../database/metadata/*.parquet').select('id', 'pubName', 'name', 'artType', 'pubDate', 'artCategory', 'pdfPage')
df = textos.join(metadata, on='id', how='inner')
del textos, metadata

In [8]:
df = (
    df
    .with_columns(pl.col('pubDate').str.to_date(format='%d/%m/%Y'))
    .filter(
        pl.col('artCategory').str.to_lowercase().str.contains(r'(marinha|ex[ée]rcito|aeron[áa]utica)'), 
        pl.col('pubDate').dt.year() >= 2018,
        pl.col('artCategory').str.to_lowercase().str.starts_with('ministério da defesa')
    )
)

In [12]:
from collections import Counter

categorias = (
    df
    .select('artCategory')
    .drop_nulls()
    .to_series()
    .to_list()
)

contagens = Counter()

for categoria in categorias:
    partes = [parte.strip() for parte in categoria.split('/') if parte.strip()]
    for nivel in range(len(partes)):
        no = ' / '.join(partes[: nivel + 1])
        contagens[no] += 1

raiz = 'Organizações Militares'
ids = [raiz]
labels = [raiz]
parents = ['']
values = [len(categorias)]

for no, valor in sorted(contagens.items(), key=lambda item: (item[0].count(' / '), item[0])):
    parent = no.rsplit(' / ', 1)[0] if ' / ' in no else raiz
    ids.append(no)
    labels.append(no.split(' / ')[-1])
    parents.append(parent)
    values.append(valor)

fig = go.Figure(
    go.Treemap(
        ids=ids,
        labels=labels,
        parents=parents,
        values=values,
        branchvalues='total',
        textinfo='label+value+percent parent',
        hovertemplate='%{id}<br>%{value} registros<br>%{percentParent:.1%} do nível acima<extra></extra>',
        maxdepth=4,
        pathbar=dict(visible=True),
    )
)

fig.update_layout(
    title='Hierarquia institucional das Forças Armadas, segundo o volume de publicações no DOU',
    margin=dict(t=50, l=20, r=20, b=20),
    height=900,
)

fig.show()


In [ ]:
from collections import Counter

base_treemap = (
    df
    .with_columns(
        forca=pl.when(pl.col('artCategory').str.to_lowercase().str.contains('comando do exército'))
        .then(pl.lit('Comando do Exército'))
        .when(pl.col('artCategory').str.to_lowercase().str.contains('comando da marinha'))
        .then(pl.lit('Comando da Marinha'))
        .when(pl.col('artCategory').str.to_lowercase().str.contains('comando da aeronáutica'))
        .then(pl.lit('Comando da Aeronáutica'))
        .otherwise(None),
        pubName=pl.col('pubName').str.head(3)
    )
    .select('forca', 'pubName', 'artType')
    .drop_nulls()
    .group_by('forca', 'pubName', 'artType')
    .len()
    .rename({'len': 'qtd'})
    .sort('forca', 'pubName', 'artType')
)

raiz = 'Forças Armadas'
ids = [raiz]
labels = [raiz]
parents = ['']
values = [int(base_treemap['qtd'].sum())]

contagens = Counter()

for forca, pub_name, art_type, qtd in base_treemap.iter_rows():
    contagens[(forca,)] += qtd
    contagens[(forca, pub_name)] += qtd
    contagens[(forca, pub_name, art_type)] += qtd

for chave, valor in sorted(contagens.items(), key=lambda item: (len(item[0]), item[0])):
    node_id = ' / '.join(chave)
    parent_id = ' / '.join(chave[:-1]) if len(chave) > 1 else raiz
    ids.append(node_id)
    labels.append(chave[-1])
    parents.append(parent_id)
    values.append(valor)

fig = go.Figure(
    go.Treemap(
        ids=ids,
        labels=labels,
        parents=parents,
        values=values,
        branchvalues='total',
        textinfo='label+value+percent parent',
        hovertemplate='%{id}<br>%{value} registros<br>%{percentParent:.1%} do nível acima<extra></extra>',
        maxdepth=3,
        pathbar=dict(visible=True),
    )
)

fig.update_layout(
    title='Comandos militares por seção do DOU e tipo de documento',
    margin=dict(t=50, l=20, r=20, b=20),
    height=900,
)

fig.show()


In [24]:
df.head()

id,texto,pubName,name,artType,pubDate,artCategory,pdfPage
str,str,str,str,str,date,str,str
"""515_20190102_11355821""",""" PORTARIA Nº 2.082, DE 27 DE D…","""DO1""","""Port_2082""","""Portaria""",2019-01-02,"""Ministério da Defesa/Comando d…","""http://pesquisa.in.gov.br/impr…"
"""515_20190102_11355822""",""" PORTARIA Nº 2.083, DE 27 DE D…","""DO1""","""Port_2083""","""Portaria""",2019-01-02,"""Ministério da Defesa/Comando d…","""http://pesquisa.in.gov.br/impr…"
"""515_20190104_11366759""",""" PORTARIA Nº 401/DPC, DE 19 DE…","""DO1""","""Port-401-2018-DPC-ALTNOR-001-M…","""Portaria""",2019-01-04,"""Ministério da Defesa/Comando d…","""http://pesquisa.in.gov.br/impr…"
"""515_20190107_11365668-1""",""" PORTARIA DECEA Nº 258/JJAER,…","""DO1""","""20181210_Portaria_258 JJAER_RJ…","""Portaria""",2019-01-07,"""Ministério da Defesa/Comando d…","""http://pesquisa.in.gov.br/impr…"
"""515_20190107_11365668-2""",""" Art. 38 As sessões serão públ…","""DO1""","""20181210_Portaria_258 JJAER_RJ…","""Portaria""",2019-01-07,"""Ministério da Defesa/Comando d…","""http://pesquisa.in.gov.br/impr…"


In [34]:
(
    df
    .filter(
        pl.col('pubName').str.head(3)=='DO3'
    )
    .group_by(
        'artType'
    )
    .agg(
        pl.len()/len(df.filter(pl.col('pubName').str.head(3)=='DO3'))
    )
    .filter(
        pl.col('len') >= 0.001
    )
    .sort(
        by='len',
        descending=True
    )
    .show(
        limit=-1
    )
)

artType,len
str,f64
"""Extrato de Termo Aditivo""",0.205673
"""Extrato de Contrato""",0.172981
"""Aviso de Licitação-Pregão""",0.169205
"""Resultado de Julgamento""",0.055767
"""Extrato de Inexigibilidade de …",0.049923
"""Extrato de Credenciamento""",0.037989
"""Extrato de Registro de Preços""",0.025392
"""Aviso""",0.025242
"""Edital de Notificação""",0.021384


In [68]:
print(
    df.filter(
        pl.col('id')== "515_20200103_12328732-2"	
    ).item(0, 'texto')
)


<p>
2868 - Combustíveis e Lubrificantes de Aviação.
Diretor do Centro Logístico da Aeronáutica (CELOG).
Não se aplica.
Não se aplica.
7U72 - Adequação, Revitalização e Modernização da Frota de Aeronaves AM-X (Projeto A-1M).
Presidente da Comissão Coordenadora do Programa Aeronave de Combate (COPAC).
Não se aplica.
Não se aplica.
PROGRAMA: 2108 - Programa de Gestão e Manutenção do Ministério da Defesa
Ação Orçamentária
Cargo/Função
Plano Orçamentário (PO)
Cargo/Função
09HB - Contribuição da União, de suas Autarquias e Fundações para o Custeio do Regime de Previdência dos Servidores Públicos Federais.
Subdiretor de Pagamento de Pessoal (SDPP) da Diretoria de Administração (DIRAD).
Não se aplica.
Não se aplica.
15F1 - Disponibilização de Pró-prios Nacionais Residenciais para os Comandos Militares.
Chefe da Quarta Subchefia (4SC) do Estado-Maior da Aeronáutica (EMAER).
0003 - Reforma de Imóveis.
Chefe da Quarta Subchefia (4SC) do Estado-Maior da Aeronáutica (EMAER).
2E74 - Estruturação e 

In [80]:
(
    df
    .filter(pl.col('pubName') == 'DO1', pl.col('artType')=='Portaria')
    .with_columns(pl.col('texto').str.replace_all(r'<\/?p>', '').str.split('\n').list.slice(1, 1).list.first())
    .select('texto')
    .drop_nulls()
    .filter(pl.col('texto')!='')
    .with_columns(pl.col('texto').str.strip_chars(r'[\(\)\*\s]+').str.split(' ').list.slice(0, 5).list.join(' '))
    .group_by('texto')
    .agg(pl.len())
    .sort(by='len', descending=True)
).show(limit=2000)

texto,len
str,u32
"""O DIRETOR DO INSTITUTO DE""",282
"""Dispensa da obrigatoriedade do…",145
"""Aprova o Aviso de Convocação""",120
"""Prorroga o prazo do credenciam…",101
"""O DIRETOR DO HOSPITAL NAVAL""",100
"""Altera as Normas da Autoridade""",99
"""O DIRETOR DE ADMINISTRAÇÃO DA""",68
"""Declara o caráter militar das""",61
"""O COMANDANTE DA AERONÁUTICA, n…",60


In [ ]:
# df.filter(pl.col('pubName')=='DO1').group_by('artType').agg(pl.len()).sort('len', descending=True).show(limit=5)

In [ ]:
# dou1 = (
#     df.filter((pl.col('pubName')=='DO1') & (pl.col('artType').is_in(['Portaria', 'Despacho', 'Ata', 'Ato', 'Resolução'])) & (~pl.col('texto').str.to_lowercase().str.starts_with(r'.?retificação')))
# )

In [ ]:
# (
#     dou1
#     .with_columns(
#         requisicao=pl.col('texto').str.to_lowercase().str.contains(r'requisitar'),
#         publicacao=pl.col('texto').str.to_lowercase().str.contains(r'(publicar|tornar p[úu]blic[ao]|seja dada publicidade)'),
#         elevacao=pl.col('texto').str.to_lowercase().str.contains(r'elevar'),
#         aprovacao=pl.col('texto').str.to_lowercase().str.contains(r'(aprovar|ficam? aprovad[ao]s?|aprova [ao])'),
#         prorrogacao=pl.col('texto').str.to_lowercase().str.contains(r'prorrogar'),
#         delegacao=pl.col('texto').str.to_lowercase().str.contains(r'(delegar|delega competência|fica delegad[ao])'),
#         renovacao=pl.col('texto').str.to_lowercase().str.contains(r'renovar'),
#         cassacao=pl.col('texto').str.to_lowercase().str.contains(r'(cassar| fica cassad[ao])'),
#         comunicacao=pl.col('texto').str.to_lowercase().str.contains(r'comunicar'),
#         declaracao=pl.col('texto').str.to_lowercase().str.contains(r'declarar'),
#         divulgacao=pl.col('texto').str.to_lowercase().str.contains(r'(divulgar|divulga [ao]|fica divulgad[ao])'),
#         penalizacao=pl.col('texto').str.to_lowercase().str.contains(r'aplic(ar|ou) (penalidade|sanção)'),
#         criacao=pl.col('texto').str.to_lowercase().str.contains(r'(criar|cria [ao])'),
#         recriacao=pl.col('texto').str.to_lowercase().str.contains(r'recriar'),
#         dispensa=pl.col('texto').str.to_lowercase().str.contains(r'dispensar'),
#         suspensao=pl.col('texto').str.to_lowercase().str.contains(r'suspender'),
#         desativacao=pl.col('texto').str.to_lowercase().str.contains(r'(desativar|desativa [ao])'),
#         credenciamento=pl.col('texto').str.to_lowercase().str.contains(r'credenciar'),
#         estabelecimento=pl.col('texto').str.to_lowercase().str.contains(r'(estabelecer|estabelece [ao]|fica estabelecid[ao])'),
#         fixacao=pl.col('texto').str.to_lowercase().str.contains(r'(fixar|ficam? fixad[ao]s?|fixa ([ao]|vagas))'),
#         aplicacao=pl.col('texto').str.to_lowercase().str.contains(r'aplicar'),
#         realocacao=pl.col('texto').str.to_lowercase().str.contains(r'(realocar|realoca ([ao]s|função|cargo)|ficam? realocad[ao]s?)'),
#         alteracao=pl.col('texto').str.to_lowercase().str.contains(r'(alterar|altera ([ao]|dispositivo))'),
#         autorizacao=pl.col('texto').str.to_lowercase().str.contains(r'(autorizar|autoriz[ao] [ao])'),
#         indicacao=pl.col('texto').str.to_lowercase().str.contains(r'(indicar|indica [ao])'),
#         modificacao=pl.col('texto').str.to_lowercase().str.contains(r'(modificar|modifica [ao])'),
#         concessao=pl.col('texto').str.to_lowercase().str.contains(r'(conceder|concede [ao])'),
#         baixa=pl.col('texto').str.to_lowercase().str.contains(r'dar baixa'),
#         revogacao=pl.col('texto').str.to_lowercase().str.contains(r'(revogar|revoga [ao])'),
#         designacao=pl.col('texto').str.to_lowercase().str.contains(r'designar'),
#         transferencia=pl.col('texto').str.to_lowercase().str.contains(r'(transferir|transfere [ao])'),
#         cancelamento=pl.col('texto').str.to_lowercase().str.contains(r'(cancelamento d[aeo]|cancelar|cancela [ao])'),
#         disciplinamento=pl.col('texto').str.to_lowercase().str.contains(r'disciplina [ao]'),
#         atualizacao=pl.col('texto').str.to_lowercase().str.contains(r'atualizar'),
#         reclassificacao=pl.col('texto').str.to_lowercase().str.contains(r'reclassificar'),
#         reconhecimento=pl.col('texto').str.to_lowercase().str.contains(r'reconhecer'),
#         disposicao=pl.col('texto').str.to_lowercase().str.contains(r'dispõe sobre'),
#         manutencao=pl.col('texto').str.to_lowercase().str.contains(r'(manter|mantém criado)'),
#         distribuicao=pl.col('texto').str.to_lowercase().str.contains(r'distribuir'),
#         habilitacao=pl.col('texto').str.to_lowercase().str.contains(r'habilitar'),
#         ativacao=pl.col('texto').str.to_lowercase().str.contains(r'(ativar|ativa [ao]|fica ativad[ao])'),
#         determinacao=pl.col('texto').str.to_lowercase().str.contains(r'determinar'),
#         passa_vigorar=pl.col('texto').str.to_lowercase().str.contains(r'passa a vigorar'),
#         definicao=pl.col('texto').str.to_lowercase().str.contains(r'definir'),
#         instituicao=pl.col('texto').str.to_lowercase().str.contains(r'instituir'),
#         classificacao=pl.col('texto').str.to_lowercase().str.contains(r'classificar'),
#         recomendacao=pl.col('texto').str.to_lowercase().str.contains(r'(recomendar|recomenda-se)'),
#         restricao=pl.col('texto').str.to_lowercase().str.contains(r'restringir'),
#         agregacao=pl.col('texto').str.to_lowercase().str.contains(r'agregar'),
#         constituicao=pl.col('texto').str.to_lowercase().str.contains(r'constituir'),
#         reducao=pl.col('texto').str.to_lowercase().str.contains(r'reduzir'),
#         termo_vistoria=pl.col('texto').str.to_lowercase().str.contains(r'termo de vistoria para'),
#         requerimento=pl.col('texto').str.to_lowercase().str.contains(r'requerimento para'),
#         retificacao=(pl.col('texto').str.to_lowercase().str.contains(r'onde se lê') & pl.col('texto').str.to_lowercase().str.contains(r'leia-se')),
#         relacao=pl.col('texto').str.to_lowercase().str.contains(r'relacionar'),
#         inefetivar=pl.col('texto').str.to_lowercase().str.contains(r'tornar sem efeito'),
#         afastamento=pl.col('texto').str.to_lowercase().str.contains(r'afastar'),
#         nomeacao=pl.col('texto').str.to_lowercase().str.contains(r'nomear'),
#         ficha_cadastro=pl.col('texto').str.to_lowercase().str.contains(r'ficha cadastro de'),
#         inclusao=pl.col('texto').str.to_lowercase().str.contains(r'incluir'),
#         convalidacao=pl.col('texto').str.to_lowercase().str.contains(r'convalidar'),
#         extincao=pl.col('texto').str.to_lowercase().str.contains(r'extinguir'),
#         devolucao=pl.col('texto').str.to_lowercase().str.contains(r'devolver'),
#         promocao=pl.col('texto').str.to_lowercase().str.contains(r'promover'),
#         reajuste=pl.col('texto').str.to_lowercase().str.contains(r'reajustar'),
#         medida_emergencial=pl.col('texto').str.to_lowercase().str.contains(r'medida emergencial'),
#         insercao=pl.col('texto').str.to_lowercase().str.contains(r'inserir'),
#         ratificacao=pl.col('texto').str.to_lowercase().str.contains(r'ratificar'),
#         proibicao=pl.col('texto').str.to_lowercase().str.contains(r'(proibir|fica proibid[ao])'),
#         contratacao=pl.col('texto').str.to_lowercase().str.contains(r'contratar'),
#         inscricao=pl.col('texto').str.to_lowercase().str.contains(r'inscrever'),
#         denominacao=pl.col('texto').str.to_lowercase().str.contains(r'denominar'),
#         reintegracao=pl.col('texto').str.to_lowercase().str.contains(r'reintegrar'),
#         homologacao=pl.col('texto').str.to_lowercase().str.contains(r'homologar'),
#         retiro=pl.col('texto').str.to_lowercase().str.contains(r'retirar'),
#         regulamentacao=pl.col('texto').str.to_lowercase().str.contains(r'regulamenta,'),
#         estende=pl.col('texto').str.to_lowercase().str.contains(r'estender'),
#         consideracao=pl.col('texto').str.to_lowercase().str.contains(r'considerar'),
#         decisao=pl.col('texto').str.to_lowercase().str.contains(r'decido'),
#         licenciamento=pl.col('texto').str.to_lowercase().str.contains(r'licenciar'),
#         modelo_declaracao=pl.col('texto').str.to_lowercase().str.contains(r'declaração de'),
#         trafego=pl.col('texto').str.to_lowercase().str.contains(r'colocar em tráfego'),
#         transformacao=pl.col('texto').str.to_lowercase().str.contains(r'transformar'),
#         tornar_insub=pl.col('texto').str.to_lowercase().str.contains(r'tornar insubsistente'),
#         despacho=pl.col('texto').str.to_lowercase().str.contains(r'despacho [nd]'),
#         ata=pl.col('texto').str.to_lowercase().str.contains(r'ata [nd]'),
#         adocao=pl.col('texto').str.to_lowercase().str.contains(r'adotar'),
         
#     )
#     .filter(
#         ~pl.any_horizontal(cs.boolean())
#     )
#     .select('id', 'texto')
#     .show(limit=10, fmt_str_lengths=2000)
# )

In [ ]:
# df.filter(pl.col('pubName')=='DO2').group_by('artType').agg(pl.len()).sort('len', descending=True).show(limit=5)

In [ ]:
# dou2 = (
#     df.filter((pl.col('pubName')=='DO2') & (pl.col('artType').is_in(['Portaria', 'Ato Concessório', 'Ato'])) & (~pl.col('texto').str.to_lowercase().str.starts_with(r'.?retificação')))
# )

In [ ]:
# (
#     dou2
#     .with_columns(
#         nomeacao=(pl.col('texto').str.to_lowercase().str.contains(r'nomear')),
#         exoneracao=(pl.col('texto').str.to_lowercase().str.contains(r'exonerar')),
#         concessao=(pl.col('texto').str.to_lowercase().str.contains(r'concede+r')),
#         promocao=(pl.col('texto').str.to_lowercase().str.contains(r'promover')),
#         prorrogacao=(pl.col('texto').str.to_lowercase().str.contains(r'prorrogar')),
#         colocacao=(pl.col('texto').str.to_lowercase().str.contains(r'(colocar|coloca [ao])')),
#         designacao=(pl.col('texto').str.to_lowercase().str.contains(r'(designar|design[ao] [ao])')),
#         manutencao=(pl.col('texto').str.to_lowercase().str.contains(r'manter')),
#         aposentadoria=(pl.col('texto').str.to_lowercase().str.contains(r'aposentar')),
#         demissao=(pl.col('texto').str.to_lowercase().str.contains(r'demitir')),
#         contratacao=(pl.col('texto').str.to_lowercase().str.contains(r'(contratar|contrata [ao])')),
#         reversao=(pl.col('texto').str.to_lowercase().str.contains(r'(reverter|reverto [àao])')),
#         alteracao=(pl.col('texto').str.to_lowercase().str.contains(r'alterar')),
#         transferencia=(pl.col('texto').str.to_lowercase().str.contains(r'transferir')),
#         renovacao=(pl.col('texto').str.to_lowercase().str.contains(r'(renovar|renova [ao])')),
#         reforma=(pl.col('texto').str.to_lowercase().str.contains(r'(reformar|reform[ao] [ao])')),
#         garantia=(pl.col('texto').str.to_lowercase().str.contains(r'assegurar')),
#         dispensa=(pl.col('texto').str.to_lowercase().str.contains(r'dispensar')),
#         cancelamento=(pl.col('texto').str.to_lowercase().str.contains(r'cancelar')),
#         sem_efeito=(pl.col('texto').str.to_lowercase().str.contains(r'tornar? sem efeito')),
#         anulacao=(pl.col('texto').str.to_lowercase().str.contains(r'(anular|anulo [oa])')),
#         consideracao=(pl.col('texto').str.to_lowercase().str.contains(r'considerar')),
#         agregacao=(pl.col('texto').str.to_lowercase().str.contains(r'agregar')),
#         licenciamento=(pl.col('texto').str.to_lowercase().str.contains(r'licenciar')),
#         inclusao=(pl.col('texto').str.to_lowercase().str.contains(r'incluir')),
#         incorporacao=(pl.col('texto').str.to_lowercase().str.contains(r'incorporar')),
#         declaracao=(pl.col('texto').str.to_lowercase().str.contains(r'declarar')),
#         revogacao=(pl.col('texto').str.to_lowercase().str.contains(r'revogar')),
#         exclusao=(pl.col('texto').str.to_lowercase().str.contains(r'excluir')),
#         autorizacao=(pl.col('texto').str.to_lowercase().str.contains(r'autorizar')),
#         convocacao=(pl.col('texto').str.to_lowercase().str.contains(r'convocar')),
#         tornar_insub=(pl.col('texto').str.to_lowercase().str.contains(r'tornar insubsistente')),
#         suspencao=(pl.col('texto').str.to_lowercase().str.contains(r'suspender')),
#         reestabelecimento=(pl.col('texto').str.to_lowercase().str.contains(r'ree?stabelecer')),
#         cassacao=(pl.col('texto').str.to_lowercase().str.contains(r'cassar')),
#         dispor=(pl.col('texto').str.to_lowercase().str.contains(r'passar [àa] disposição')),
#         conversao=(pl.col('texto').str.to_lowercase().str.contains(r'converter')),
#         cessao2=(pl.col('texto').str.to_lowercase().str.contains(r'\bceder\b')),
#         encerramento=(pl.col('texto').str.to_lowercase().str.contains(r'encerrar')),
#         fixacao=(pl.col('texto').str.to_lowercase().str.contains(r'(fixar|fixa [ao])')),
#         desligamento=(pl.col('texto').str.to_lowercase().str.contains(r'desligar')),
#         substituicao=(pl.col('texto').str.to_lowercase().str.contains(r'substituir')),
#         retroacao=(pl.col('texto').str.to_lowercase().str.contains(r'retroagir')),
#         efetivacao=(pl.col('texto').str.to_lowercase().str.contains(r'efetivar')),
#         publicar=(pl.col('texto').str.to_lowercase().str.contains(r'tornar(.+)p[úu]blic[ao]')),
#         consignacao=(pl.col('texto').str.to_lowercase().str.contains(r'consignar')),
#         reinicio=(pl.col('texto').str.to_lowercase().str.contains(r'reiniciar')),
#         acolhimento=(pl.col('texto').str.to_lowercase().str.contains(r'acolher')),
#         aplicacao=(pl.col('texto').str.to_lowercase().str.contains(r'aplicar')),
#         reintegracao=(pl.col('texto').str.to_lowercase().str.contains(r'reintegrar')),
#         interrupcao=(pl.col('texto').str.to_lowercase().str.contains(r'interromper')),
#         deferimento=(pl.col('texto').str.to_lowercase().str.contains(r'\bdeferir\b')),
#         indeferimento=(pl.col('texto').str.to_lowercase().str.contains(r'indeferir')),
#         remocao=(pl.col('texto').str.to_lowercase().str.contains(r'remover')),
#         assuncao=(pl.col('texto').str.to_lowercase().str.contains(r'assumir')),
#         homologacao=(pl.col('texto').str.to_lowercase().str.contains(r'homologar')),
#         aprovacao=(pl.col('texto').str.to_lowercase().str.contains(r'aprovar')),
#         instituicao=(pl.col('texto').str.to_lowercase().str.contains(r'instituir')),
#         cessao1=(pl.col('texto').str.to_lowercase().str.contains(r'\bcessar\b')),
#         delegacao=(pl.col('texto').str.to_lowercase().str.contains(r'delegar')),
#         disponibilizacao=(pl.col('texto').str.to_lowercase().str.contains(r'disponibilizar')),
#         ratificacao=(pl.col('texto').str.to_lowercase().str.contains(r'ratificar')),
#         reconducao=(pl.col('texto').str.to_lowercase().str.contains(r'reconduzir')),
#         lotacao=(pl.col('texto').str.to_lowercase().str.contains(r'lotar')),
#         encostamento=(pl.col('texto').str.to_lowercase().str.contains(r'encostar')),
#         reabilitacao=(pl.col('texto').str.to_lowercase().str.contains(r'reabilitar')),
#         determinacao=(pl.col('texto').str.to_lowercase().str.contains(r'determinar')),
#         classificacao=(pl.col('texto').str.to_lowercase().str.contains(r'classificar')),
#         estabelecimento=(pl.col('texto').str.to_lowercase().str.contains(r'estabelecer')),
#         notificacao=(pl.col('texto').str.to_lowercase().str.contains(r'notificar')),
#         credenciamento=(pl.col('texto').str.to_lowercase().str.contains(r'credenciar')),
#         constituicao=(pl.col('texto').str.to_lowercase().str.contains(r'constituir')),
#         enquadramento=(pl.col('texto').str.to_lowercase().str.contains(r'enquadrar')),
#         revisao=(pl.col('texto').str.to_lowercase().str.contains(r'\brever\b')),
#         atribuicao=(pl.col('texto').str.to_lowercase().str.contains(r'atribuir')),
#         retificacao=(pl.col('texto').str.to_lowercase().str.contains(r'(retificar|onde se[\s\-]l[êe])')),
#         empossamento=(pl.col('texto').str.to_lowercase().str.contains(r'empossar')),
#         criacao=(pl.col('texto').str.to_lowercase().str.contains(r'criar')),
#         rescisao=(pl.col('texto').str.to_lowercase().str.contains(r'rescindir')),
#         reconhecimento=(pl.col('texto').str.to_lowercase().str.contains(r'reconhecer')),
#         retorno=(pl.col('texto').str.to_lowercase().str.contains(r'retornar')),
#         a_pedido=(pl.col('texto').str.to_lowercase().str.contains(r'a pedido')),
#         ex_officio=(pl.col('texto').str.to_lowercase().str.contains(r'ex[\-\s]off?[íi]cio')),
#         apuracao=(pl.col('texto').str.to_lowercase().str.contains(r'apurar os fatos')),
#         expedicao=(pl.col('texto').str.to_lowercase().str.contains(r'expedir')),
#         promo_antig=(pl.col('texto').str.to_lowercase().str.contains(r'por antiguidade')),
#         promo_merit=(pl.col('texto').str.to_lowercase().str.contains(r'por merecimento')),
#         apostilamento=(pl.col('texto').str.to_lowercase().str.contains(r'apostilar')),
#         ttc=(pl.col('texto').str.to_lowercase().str.contains(r'exercer(\sa)? tarefa')),
#         dano_erario=(pl.col('texto').str.to_lowercase().str.contains(r'(dano|ressarcimento) ao er[áa]rio')),
#         ressarcimento=(pl.col('texto').str.to_lowercase().str.contains(r'ressarci(r|mento)')),
#         missao_exterior=(pl.col('texto').str.to_lowercase().str.contains(r'missão no exterior')),
#         viuva=(pl.col('texto').str.to_lowercase().str.contains(r'vi[úu]va')),
#         passa_vigorar=(pl.col('texto').str.to_lowercase().str.contains(r'passa a vigorar')),
#         beneficiario=(pl.col('texto').str.to_lowercase().str.contains(r'beneficiári[ao]s?')),
#         extincao=(pl.col('texto').str.to_lowercase().str.contains(r'extinguir')),
#         decisao_judicial=(pl.col('texto').str.to_lowercase().str.contains(r'decisão judicial')),
#         ltip=(pl.col('texto').str.to_lowercase().str.contains(r'(licença para tratar de assunto particular|ltip)')),
#         permanencia_rr=(pl.col('texto').str.to_lowercase().str.contains(r'permanência na reserva remunerada')),
#         afastamento=(pl.col('texto').str.to_lowercase().str.contains(r'afastar')),
#         indicacao=(pl.col('texto').str.to_lowercase().str.contains(r'indicar')),
#         dilatacao=(pl.col('texto').str.to_lowercase().str.contains(r'dilatar')),
#         validacao=(pl.col('texto').str.to_lowercase().str.contains(r'\b(validar|convalidar)')),
#         habilitacao=(pl.col('texto').str.to_lowercase().str.contains(r'habilitar')),
#         divulgacao=(pl.col('texto').str.to_lowercase().str.contains(r'divulgar')),
#         alocacao=(pl.col('texto').str.to_lowercase().str.contains(r'alocar')),
#         pensao=(pl.col('texto').str.to_lowercase().str.contains(r'pens(ão|ionista)')),
#         retiro=(pl.col('texto').str.to_lowercase().str.contains(r'(retirar|retiro [ao])')),
#         auxilio=(pl.col('texto').str.to_lowercase().str.contains(r'auxílio\-')),
#     )
#     .filter(
#         ~pl.any_horizontal(cs.boolean())
#     )
#     .select('id', 'texto')
# )

In [ ]:
# df.filter(pl.col('pubName')=='DO3').group_by('artType').agg(pl.len()).sort('len', descending=True).show(limit=10)